In [0]:
%sql
-- Create the schema and a volume to land raw Foursquare files
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :schema)
  COMMENT 'Foursquare co-traveler dataset';

CREATE VOLUME IF NOT EXISTS IDENTIFIER(:catalog || '.' || :schema || '.foursquare_raw')
  COMMENT 'Raw Foursquare files downloaded from Figshare';

In [0]:
import requests
import os

ARTICLE_ID = 22126793

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
VOLUME_PATH = f"/Volumes/{catalog}/{schema}/foursquare_raw"

# Fetch file metadata from Figshare public API
resp = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE_ID}/files")
resp.raise_for_status()
files = resp.json()

print(f"Found {len(files)} file(s) in article {ARTICLE_ID}:")
for f in files:
    print(f"  {f['name']:60s}  {f['size']:>12,} bytes")

print()

# Download each file to the volume
for file_info in files:
    name = file_info["name"]
    url  = file_info["download_url"]
    dest = f"{VOLUME_PATH}/{name}"

    print(f"Downloading {name} ...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as fout:
            for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):  # 4 MB chunks
                fout.write(chunk)

    print(f"  ✓ Saved to {dest}  ({os.path.getsize(dest):,} bytes)")

print("\nAll downloads complete!")

Found 6 file(s) in article 22126793:
  dataset_WWW_Checkins_anonymized.txt                           1,543,421,675 bytes
  dataset_WWW_friendship_new.txt                                   8,745,495 bytes
  dataset_WWW_friendship_old.txt                                   5,157,271 bytes
  dataset_WWW_readme.txt                                               1,890 bytes
  raw_POIs.txt                                                   704,764,607 bytes
  raw_Checkins_anonymized.txt                                   6,110,035,393 bytes

  ✓ Saved to /Volumes/justinm_demo/cotraveler/foursquare_raw/dataset_WWW_Checkins_anonymized.txt  (1,543,421,675 bytes)
  ✓ Saved to /Volumes/justinm_demo/cotraveler/foursquare_raw/dataset_WWW_friendship_new.txt  (8,745,495 bytes)
  ✓ Saved to /Volumes/justinm_demo/cotraveler/foursquare_raw/dataset_WWW_friendship_old.txt  (5,157,271 bytes)
  ✓ Saved to /Volumes/justinm_demo/cotraveler/foursquare_raw/dataset_WWW_readme.txt  (1,890 bytes)
  ✓ Saved to /Volumes

In [0]:
import os

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
VOLUME_PATH = f"/Volumes/{catalog}/{schema}/foursquare_raw"

print(f"Files in {VOLUME_PATH}:\n")
for fname in sorted(os.listdir(VOLUME_PATH)):
    size = os.path.getsize(f"{VOLUME_PATH}/{fname}")
    print(f"  {fname:60s}  {size:>14,} bytes")

In [0]:
import os
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, IntegerType
)

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
VOLUME_PATH = f"/Volumes/{catalog}/{schema}/foursquare_raw"
CATALOG     = catalog
SCHEMA      = schema

# Schemas from dataset_WWW_readme.txt  (all files are headerless TSV)
FILE_CONFIGS = {
    "dataset_WWW_Checkins_anonymized.txt": {
        "table": "checkins_www",
        "schema": StructType([
            StructField("user_id",                  StringType(), True),
            StructField("venue_id",                 StringType(), True),
            StructField("utc_time",                 StringType(), True),
            StructField("timezone_offset_minutes",  IntegerType(), True),
        ]),
    },
    "raw_Checkins_anonymized.txt": {
        "table": "checkins_raw",
        "schema": StructType([
            StructField("user_id",                  StringType(), True),
            StructField("venue_id",                 StringType(), True),
            StructField("utc_time",                 StringType(), True),
            StructField("timezone_offset_minutes",  IntegerType(), True),
        ]),
    },
    "raw_POIs.txt": {
        "table": "pois",
        "schema": StructType([
            StructField("venue_id",             StringType(), True),
            StructField("latitude",             DoubleType(), True),
            StructField("longitude",            DoubleType(), True),
            StructField("venue_category_name",  StringType(), True),
            StructField("country_code",         StringType(), True),
        ]),
    },
    "dataset_WWW_friendship_old.txt": {
        "table": "friendship_old",
        "schema": StructType([
            StructField("user_id_1", StringType(), True),
            StructField("user_id_2", StringType(), True),
        ]),
    },
    "dataset_WWW_friendship_new.txt": {
        "table": "friendship_new",
        "schema": StructType([
            StructField("user_id_1", StringType(), True),
            StructField("user_id_2", StringType(), True),
        ]),
    },
}

for fname, cfg in FILE_CONFIGS.items():
    file_path  = f"{VOLUME_PATH}/{fname}"
    full_table = f"{CATALOG}.{SCHEMA}.{cfg['table']}"

    print(f"Reading  : {fname}")
    print(f"Target   : {full_table}")

    df = (
        spark.read
            .option("sep", "\t")
            .option("header", "false")
            .schema(cfg["schema"])
            .csv(file_path)
    )

    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table))

    row_count = spark.table(full_table).count()
    print(f"  ✓ {row_count:,} rows  |  columns: {df.columns}\n")

print("All tables created successfully!")

Reading  : dataset_WWW_Checkins_anonymized.txt
Target   : justinm_demo.cotraveler.checkins_www
  ✓ 22,809,624 rows  |  columns: ['user_id', 'venue_id', 'utc_time', 'timezone_offset_minutes']

Reading  : raw_Checkins_anonymized.txt
Target   : justinm_demo.cotraveler.checkins_raw
  ✓ 90,048,627 rows  |  columns: ['user_id', 'venue_id', 'utc_time', 'timezone_offset_minutes']

Reading  : raw_POIs.txt
Target   : justinm_demo.cotraveler.pois
  ✓ 11,180,160 rows  |  columns: ['venue_id', 'latitude', 'longitude', 'venue_category_name', 'country_code']

Reading  : dataset_WWW_friendship_old.txt
Target   : justinm_demo.cotraveler.friendship_old
  ✓ 363,704 rows  |  columns: ['user_id_1', 'user_id_2']

Reading  : dataset_WWW_friendship_new.txt
Target   : justinm_demo.cotraveler.friendship_new
  ✓ 607,333 rows  |  columns: ['user_id_1', 'user_id_2']

All tables created successfully!
